# 01 — Data understanding

**Why this notebook exists:** before modeling, we need to know *what* we have, *how big* it is, *what is missing*, and *how rare fraud is*. Skipping this step is how leakage, silent duplicates, and misleading accuracy sneak into a project.

## Dataset source

We use the **Credit Card Fraud Detection** dataset published by the [ULB Machine Learning Group](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) (Université Libre de Bruxelles). A public CSV mirror (the same file used in TensorFlow tutorials) is downloaded by `python scripts/download_data.py`.

Transactions were collected over two days. Features **V1–V28** are **PCA components** of confidential raw attributes. **Time** and **Amount** were not transformed. **Class** is the target: `1` = fraud, `0` = legitimate.

## Target variable

This is a **binary classification** problem: assign each transaction to one of two classes.

1. **What it is:** learning a mapping from features → class (or class probability).
2. **Why we need it:** reviewers need a score, not an unsupervised cluster id.
3. **How it works:** a model estimates \(P(\text{fraud} \mid x)\) and a threshold turns that into a yes/no flag.
4. **In this project:** `Class` is that label. It is extremely imbalanced.

## Limitations (read this twice)

- PCA features are **not interpretable merchant fields**.
- Two days of European card traffic ≠ your bank's 2026 stream (**distribution shift**).
- Labels can be delayed or incomplete in real life; here they are given as ground truth.
- We will **not fabricate** row counts. Run the cells on the real file.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_processing import dataset_overview, drop_duplicates_if_present, load_raw_dataset
from src.utils import RAW_DATASET_PATH


## Load the file

**Why:** if the CSV is missing, every later notebook is theater. Fail loudly with a path the user can fix.


In [ ]:
from pathlib import Path
print("Expected CSV:", RAW_DATASET_PATH)
print("Exists:", RAW_DATASET_PATH.exists())
df = load_raw_dataset()
df.head()


## Shape, columns, dtypes

**Why:** shape tells us whether we can afford heavy models; dtypes tell us what must be encoded or scaled; column names tell us what we are *not* allowed to treat as causal (the `V*` fields).


In [ ]:
print("shape:", df.shape)
print("columns:", list(df.columns))
df.dtypes


## Descriptive statistics

**Why:** `Amount` is a heavy-tailed money field; `Time` is a clock in seconds from the first event; `V*` should look roughly centered because they are PCA scores. We are looking for impossible negatives on Amount, empty columns, or surprise non-numeric types — not for a story we invented before seeing the data.


In [ ]:
df.describe().T


## Missing values

**Why:** sklearn estimators generally refuse `NaN`. If missingness is related to fraud, dropping rows could bias the label rate. This public file is known to be complete — we still **measure** it instead of assuming.


In [ ]:
missing = df.isna().sum()
print("total missing cells:", int(missing.sum()))
missing[missing > 0]


## Duplicate records

**Why:** an identical row in both train and test is a form of **leakage**. The model "sees" the test transaction during training. Exact duplicates are not independent extra evidence.


In [ ]:
n_dup = int(df.duplicated().sum())
print("duplicate rows:", n_dup)
print("duplicate rate:", n_dup / len(df))


## Class distribution — the most important table in this project

**Why:** if 99.8% of rows are legitimate, **accuracy is the wrong headline metric**. A model that predicts "legitimate" every time would look almost perfect and catch **zero fraud**.


In [ ]:
overview = dataset_overview(df)
overview


In [ ]:
counts = df["Class"].value_counts()
print(counts)
print("legitimate %:", 100 * counts.get(0, 0) / len(df))
print("fraud %:", 100 * counts.get(1, 0) / len(df))


## Accuracy trap (toy arithmetic)

Suppose 99.8% of transactions are legitimate and 0.2% are fraud.

A constant "not fraud" classifier:

- accuracy ≈ 0.998
- recall for fraud = 0
- precision for fraud is undefined or 0

**That is why later notebooks optimize PR-AUC / recall / precision, not accuracy.**

## What we take into preprocessing

- Drop exact duplicates (justified: copies, not new events).
- Stratified train/test split so both sides keep the tiny fraud rate.
- Scale numeric fields **after** the split, fit on train only.
